# 🏙️ Populous Cities Analysis

## Overview
This notebook contains a practice exercise completed during the AI Bootcamp using **Pandas** and **NumPy**.

## Objective
The goal of this exercise is to explore and manipulate a dataset containing information about more than **40,000 cities**. The notebook focuses on developing practical data analysis skills by performing common operations on a Pandas DataFrame.

## Skills Practiced
- Reading and exploring datasets
- DataFrame indexing and selection
- Filtering and sorting data
- Data manipulation with Pandas
- Numerical operations with NumPy

---
**Tools:** Python, Pandas, NumPy

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_excel("worldcities.xlsx")
df.head()

,ville,ville_ascii,lat,lng,pays,iso2,iso3,admin_nom,capital,population,id
0,A Coruña,A Coruna,43.3667,-8.3833,Spain,ES,ESP,Galicia,minor,245468.0,1.724417e+09
1,A Yun Pa,A Yun Pa,13.3939,108.4408,Vietnam,VN,VNM,Gia Lai,minor,53720.0,1.704946e+09
2,Aabenraa,Aabenraa,55.0444,9.4181,Denmark,DK,DNK,Syddanmark,minor,16401.0,1.208000e+09
3,Aachen,Aachen,50.7756,6.0836,Germany,DE,DEU,North Rhine-Westphalia,minor,249070.0,1.276806e+09
4,Aadorf,Aadorf,47.4939,8.8975,Switzerland,CH,CHE,Thurgau,NaN,9036.0,1.756023e+09


In [4]:
# drop and rename
df = df.drop(columns=["id", "capital", "ville_ascii", "admin_nom"])
df = df.rename(columns={"ville": "city","pays": "country"})
df.head()

,city,lat,lng,country,iso2,iso3,population
0,A Coruña,43.3667,-8.3833,Spain,ES,ESP,245468.0
1,A Yun Pa,13.3939,108.4408,Vietnam,VN,VNM,53720.0
2,Aabenraa,55.0444,9.4181,Denmark,DK,DNK,16401.0
3,Aachen,50.7756,6.0836,Germany,DE,DEU,249070.0
4,Aadorf,47.4939,8.8975,Switzerland,CH,CHE,9036.0


In [5]:
# filter data
df = df[df["population"] >= 1_000_000]
df.shape

(776, 7)

In [6]:
# change type population
df = df.astype({"population": np.int32})
df.dtypes

city           object
lat           float64
lng           float64
country        object
iso2           object
iso3           object
population      int32
dtype: object

In [8]:
first_df = df.copy()

In [16]:
# remove duplicated and missed values
df = df.drop_duplicates()
df = df.dropna(thresh=df.shape[1] - 1)
df.shape

(767, 7)

In [19]:
# fill missing values of 'lat' and 'lng' columns
df["lat"] = df.groupby("country")["lat"].transform(lambda x: x.fillna(x.mean()))
df["lng"] = df.groupby("country")["lng"].transform(lambda x: x.fillna(x.mean()))

In [22]:
def haversine_from_teh(lat, lng):
    tehran = df[df["city"] == "Tehran"].iloc[0]
    lat_teh = np.radians(tehran["lat"])
    lng_teh = np.radians(tehran["lng"])
    lat = np.radians(lat)
    lng = np.radians(lng)
    dlat = lat - lat_teh
    dlng = lng - lng_teh
    a = np.sin(dlat/2)**2 + np.cos(lat_teh) * np.cos(lat) * np.sin(dlng/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    R = 6371
    d = R * c 
    return d 

In [24]:
# create distance_from_tehran column
df["distance_from_tehran"] = df.apply(lambda row: haversine_from_teh(row["lat"], row["lng"]), axis=1)

In [26]:
# sort and save dataframe df 
df = df.sort_values(by=["city", "lat"], ascending=[True, False])
df.to_csv("distances.csv", index=False)

In [30]:
# sort and save dataframe first_df
first_df = first_df.sort_values(by=["city", "lat"], ascending=[True, False])
first_df.to_csv("cities.csv", index=False)